# R1 Timetable Comparison
The goal of this notebook is to compare `actual_departure` values from the R1 train line against the official schedules (`R1_direction1_schedules` and `R1_direction2_schedules`).
We will evaluate if it's viable to assign each train its scheduled time by calculating matching errors and plotting viability metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
tqdm.pandas()

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load official schedules
dir1_schedule = pd.read_csv('official_timetables/R1_direction1_schedules.csv')
dir2_schedule = pd.read_csv('official_timetables/R1_direction2_schedules.csv')

# Filter to workdays to match the Monday parquet file
dir1_schedule = dir1_schedule[dir1_schedule['Day_Type'].str.contains('Workdays|Laborables', case=False, na=False)].copy()
dir2_schedule = dir2_schedule[dir2_schedule['Day_Type'].str.contains('Workdays|Laborables', case=False, na=False)].copy()

In [ ]:
# Load parquet files (Weekday: 2026-03-16 Monday)
wk_parquet_path = '../data/dynamic/timetables/timetables_2026_03_16.parquet'
df_weekday = pd.read_parquet(wk_parquet_path)

# Drop the planned columns as we only care about actual departures
df_weekday = df_weekday.drop(columns=['planned_arrival', 'planned_departure'], errors='ignore')

## Mapping Stations and Filtering R1 Trains
We merge `station_id` with station names to identify trains going in Direction 1 vs Direction 2.

In [ ]:
# Load stations data
stations_df = pd.read_parquet('../data/static/stations.parquet')

# Map station names to the weekday dataframe
df_weekday = df_weekday.merge(stations_df[['station_id', 'name']], on='station_id', how='left')

dir1_stations = dir1_schedule.columns.tolist()[:-1] # Exclude 'Day_Type'
dir2_stations = dir2_schedule.columns.tolist()[:-1] # Exclude 'Day_Type'

# Filter the parquet for only the stations that belong to R1
r1_all_stations = list(set(dir1_stations + dir2_stations))
df_r1 = df_weekday[df_weekday['name'].isin(r1_all_stations)].copy()

# Convert actual_departure to datetime and create the format time float
df_r1['departure_time'] = pd.to_datetime(df_r1['actual_departure'], errors='coerce')
df_r1['schedule_format_time'] = df_r1['departure_time'].dt.hour + (df_r1['departure_time'].dt.minute / 100.0)

# Drop records with NaT departure times
df_r1 = df_r1.dropna(subset=['schedule_format_time'])


In [ ]:
# Tag train directions
def determine_direction(group):
    group = group.sort_values('stop_sequence')
    stations = group['name'].dropna().tolist()
    if len(stations) < 2: return 'Unknown'
    
    for i in range(len(stations)):
        for j in range(i+1, len(stations)):
            s1, s2 = stations[i], stations[j]
            if s1 in dir1_stations and s2 in dir1_stations:
                idx1 = dir1_stations.index(s1)
                idx2 = dir1_stations.index(s2)
                if idx1 < idx2:
                    return 'Direction 1'
                elif idx1 > idx2:
                    return 'Direction 2'
    return 'Unknown'

try:
    train_directions = df_r1.groupby('train_id').apply(determine_direction, include_groups=False).reset_index(name='direction')
except TypeError:
    train_directions = df_r1.groupby('train_id').apply(lambda g: determine_direction(g)).reset_index(name='direction')

df_r1 = df_r1.merge(train_directions, on='train_id')
print('Trains by direction:')
print(df_r1.groupby('train_id')['direction'].first().value_counts())


## Batch Matching All Trains to the Timetables
We calculate the Mean Absolute Error (MAE) for every train against every valid schedule row to find the best match.

In [ ]:
def clean_schedule_time(val):
    if pd.isna(val): return np.nan
    if isinstance(val, str):
        try: return float(val.replace(',', '.'))
        except: return np.nan
    return float(val)

for col in dir1_stations:
    if col in dir1_schedule.columns:
        dir1_schedule[col] = dir1_schedule[col].apply(clean_schedule_time)
        
for col in dir2_stations:
    if col in dir2_schedule.columns:
        dir2_schedule[col] = dir2_schedule[col].apply(clean_schedule_time)

def find_best_match(actual_series, schedule_df):
    # Compute difference matrix
    diff_matrix = []
    for idx, sched_row in schedule_df.iterrows():
        diffs = []
        for station in actual_series.index:
            if pd.notna(actual_series.get(station)) and pd.notna(sched_row.get(station)):
                diff = abs(actual_series[station] - sched_row[station])
                # Handle wraparound at midnight roughly
                if diff > 12: diff = 24 - diff
                diffs.append(diff)
        if diffs:
            diff_matrix.append((idx, np.mean(diffs), len(diffs)))
            
    if diff_matrix:
        # Sort by MAE, break ties by number of matching stations
        best_match = min(diff_matrix, key=lambda x: x[1])
        return best_match[0], best_match[1]
    return None, None

match_results = []
station_errors = []

# Process Direction 1
dir1_trains = df_r1[df_r1['direction'] == 'Direction 1'].drop_duplicates(subset=['train_id', 'name'])
dir1_pivot = dir1_trains.pivot(index='train_id', columns='name', values='schedule_format_time')

print("Matching Direction 1 trains...")
for train_id, row in tqdm(dir1_pivot.iterrows(), total=len(dir1_pivot)):
    best_idx, mae = find_best_match(row, dir1_schedule)
    if best_idx is not None:
        match_results.append({'train_id': train_id, 'direction': 'Direction 1', 'schedule_idx': best_idx, 'mae': mae})
        
        # Record individual station errors
        sched_row = dir1_schedule.loc[best_idx]
        for station in row.index:
            if pd.notna(row.get(station)) and pd.notna(sched_row.get(station)):
                 err = row[station] - sched_row[station]
                 # Convert hour float error roughly to minutes
                 err_minutes = err * 60
                 station_errors.append({'train_id': train_id, 'station': station, 'direction': 'Direction 1', 
                                        'actual': row[station], 'scheduled': sched_row[station], 'error_minutes': err_minutes})

# Process Direction 2
dir2_trains = df_r1[df_r1['direction'] == 'Direction 2'].drop_duplicates(subset=['train_id', 'name'])
dir2_pivot = dir2_trains.pivot(index='train_id', columns='name', values='schedule_format_time')

print("Matching Direction 2 trains...")
for train_id, row in tqdm(dir2_pivot.iterrows(), total=len(dir2_pivot)):
    best_idx, mae = find_best_match(row, dir2_schedule)
    if best_idx is not None:
        match_results.append({'train_id': train_id, 'direction': 'Direction 2', 'schedule_idx': best_idx, 'mae': mae})
        
        # Record individual station errors
        sched_row = dir2_schedule.loc[best_idx]
        for station in row.index:
            if pd.notna(row.get(station)) and pd.notna(sched_row.get(station)):
                 err = row[station] - sched_row[station]
                 err_minutes = err * 60
                 station_errors.append({'train_id': train_id, 'station': station, 'direction': 'Direction 2', 
                                        'actual': row[station], 'scheduled': sched_row[station], 'error_minutes': err_minutes})

matches_df = pd.DataFrame(match_results)
errors_df = pd.DataFrame(station_errors)
print(f"Total trains matched: {len(matches_df)}")


## Viability Plots
### Plot 1: Volume Comparison (Reported vs Scheduled)
Does the number of trains stopping at each station in the actual data match the number officially scheduled?

In [ ]:
# Count actual departures from df_r1
actual_counts = df_r1['name'].value_counts().rename('Actual Departures')

# Count scheduled departures
sched_counts_dir1 = dir1_schedule.count().drop('Day_Type', errors='ignore')
sched_counts_dir2 = dir2_schedule.count().drop('Day_Type', errors='ignore')
sched_counts = sched_counts_dir1.add(sched_counts_dir2, fill_value=0).rename('Scheduled Departures')

volume_df = pd.concat([actual_counts, sched_counts], axis=1).fillna(0)

volume_df.sort_values('Scheduled Departures', ascending=True).plot(kind='barh', figsize=(10, 12), width=0.8)
plt.title('Reported Departures vs Scheduled Departures per Station')
plt.xlabel('Number of Trains')
plt.ylabel('Station')
plt.show()

### Plot 2: Error Distribution (MAE per Train)
Shows the distribution of average match error across all trains. An MAE of 0.10 roughly equals 6 minutes of error.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(matches_df['mae'] * 60, bins=50, kde=True)
plt.title('Distribution of Match Error per Train')
plt.xlabel('Mean Absolute Error (Approx. Minutes)')
plt.ylabel('Number of Trains')
plt.axvline((matches_df['mae'] * 60).mean(), color='red', linestyle='--', label=f"Mean Error: {(matches_df['mae'] * 60).mean():.1f} mins")
plt.legend()
plt.show()

### Plot 3: Station-Level Errors
Are some stations systematically delayed compared to their schedule?

In [ ]:
plt.figure(figsize=(14, 8))
# Reorder stations based on median error to spot trends
order = errors_df.groupby('station')['error_minutes'].median().sort_values().index

sns.boxplot(x='station', y='error_minutes', data=errors_df, order=order, showfliers=False, color='lightblue')
plt.xticks(rotation=90)
plt.axhline(0, color='red', linestyle='--')
plt.title('Distribution of Time Differences (Actual - Scheduled) per Station')
plt.ylabel('Delay/Error (Minutes)')
plt.xlabel('Station')
plt.tight_layout()
plt.show()